# Selection Sort and In-Place Minima Selection

**Session:** 2026-10-28 · Brute-force Exhaustive  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

**Selection Sort** is a canonical brute-force sorting algorithm. It proceeds by repeatedly locating the minimum element from the unsorted suffix and swapping it into its sorted final position at the prefix.

## Learning Outcomes

- Implement Selection Sort from first principles with full type hints.
- Maintain and prove loop invariants during execution.
- Formally derive the exact comparison formula: $\frac{n(n-1)}{2}$.
- Analyze memory write optimality ($O(n)$ swaps) and stability implications.


## 1. Algorithm Mechanics and Implementation

Selection sort maintains two sub-arrays within the input array $A$:
1. A sorted sub-array at indices $[0 \dots \text{step} - 1]$.
2. An unsorted sub-array at indices $[\text{step} \dots n - 1]$.


In [ ]:
from typing import TypeVar

T = TypeVar("T")


def selection_sort(array: list[T]) -> list[T]:
    """Sort an array in-place using Selection Sort.

    Args:
        array: List of comparable items.

    Returns:
        The sorted list in non-decreasing order.
    """
    n = len(array)
    for step in range(n):
        min_idx = step
        for i in range(step + 1, n):
            if array[i] < array[min_idx]:
                min_idx = i

        array[step], array[min_idx] = array[min_idx], array[step]

    return array


# Test with the lecture example
test_arr = [7, 2, 5, 4, 17]
result = selection_sort(test_arr.copy())
print("Sorted result:", result)
assert result == [2, 4, 5, 7, 17]

## 2. Step-by-Step Execution Trace

Let us trace `[7, 2, 5, 4, 17]` across each outer step:

```text
Initial Array: [7, 2, 5, 4, 17]
Step 0: Min among [7, 2, 5, 4, 17] is 2 at idx 1 -> Swap 7 and 2 -> [2, 7, 5, 4, 17]
Step 1: Min among [7, 5, 4, 17] is 4 at idx 3    -> Swap 7 and 4 -> [2, 4, 5, 7, 17]
Step 2: Min among [5, 7, 17] is 5 at idx 2       -> In place     -> [2, 4, 5, 7, 17]
Step 3: Min among [7, 17] is 7 at idx 3          -> In place     -> [2, 4, 5, 7, 17]
Final:  [2, 4, 5, 7, 17]
```


In [ ]:
def trace_selection_sort(arr: list[int]) -> list[tuple[int, list[int], int]]:
    """Record state trace at each step of selection sort."""
    history = []
    a = arr.copy()
    n = len(a)
    for step in range(n - 1):
        min_idx = step
        for i in range(step + 1, n):
            if a[i] < a[min_idx]:
                min_idx = i
        a[step], a[min_idx] = a[min_idx], a[step]
        history.append((step, a.copy(), min_idx))
    return history


for step, state, chosen_min in trace_selection_sort([7, 2, 5, 4, 17]):
    print(f"Step {step}: min element selected from index {chosen_min} -> Array: {state}")

## 3. Mathematical Derivation of Comparison Count

At outer loop index `step` ($k = 0, 1, \dots, n - 2$):
- Number of inner loop iterations = $n - 1 - k$ comparisons.

Total comparisons:
$$C(n) = \sum_{k=0}^{n-2} (n - 1 - k) = (n - 1) + (n - 2) + \dots + 1 = \frac{n(n - 1)}{2} = \frac{n^2 - n}{2}$$

> [!NOTE]
> The original lecture slide had an arithmetic typo writing $\frac{n(n-2)}{2}$. The rigorous closed-form sum of the first $n-1$ positive integers is unambiguously $\frac{n(n-1)}{2}$.

### Complexity Summary

| Metric | Best Case | Average Case | Worst Case |
| :--- | :--- | :--- | :--- |
| Comparisons | $\frac{n(n-1)}{2} = \Theta(n^2)$ | $\frac{n(n-1)}{2} = \Theta(n^2)$ | $\frac{n(n-1)}{2} = \Theta(n^2)$ |
| Swaps (Writes) | $0$ to $n-1 = O(n)$ | $n - 1 = O(n)$ | $n - 1 = O(n)$ |
| Auxiliary Space | $O(1)$ | $O(1)$ | $O(1)$ |

Selection Sort is particularly advantageous when write operations are orders of magnitude more expensive than reads (e.g., flash EEPROM memory).


In [ ]:
def verify_comparisons_and_swaps(n: int):
    arr = list(range(n, 0, -1))
    comps = 0
    swaps = 0
    for step in range(n):
        min_idx = step
        for i in range(step + 1, n):
            comps += 1
            if arr[i] < arr[min_idx]:
                min_idx = i
        if min_idx != step:
            swaps += 1
            arr[step], arr[min_idx] = arr[min_idx], arr[step]

    expected_comps = n * (n - 1) // 2
    assert comps == expected_comps
    return comps, swaps


for size in [5, 10, 20]:
    c, s = verify_comparisons_and_swaps(size)
    print(f"Size {size:2d}: Comparisons = {c} (matches n(n-1)/2), Swaps = {s}")

## 4. Stability Analysis

A sorting algorithm is **stable** if elements with identical keys retain their original relative order.

Selection Sort is inherently **unstable**. For example, sorting `[(2, 'A'), (2, 'B'), (1, 'C')]`:
- Step 0 selects `(1, 'C')` and swaps with `(2, 'A')`.
- Result puts `(2, 'B')` before `(2, 'A')`, reversing their initial order.


In [ ]:
def is_selection_sort_stable() -> bool:
    items = [(2, "first_2"), (2, "second_2"), (1, "one")]
    # Sort by primary integer key
    n = len(items)
    for step in range(n):
        min_idx = step
        for i in range(step + 1, n):
            if items[i][0] < items[min_idx][0]:
                min_idx = i
        items[step], items[min_idx] = items[min_idx], items[step]

    # Check if 'first_2' is still before 'second_2'
    labels = [tag for _, tag in items if _ == 2]
    return labels == ["first_2", "second_2"]


print("Is standard Selection Sort stable?", is_selection_sort_stable())
assert is_selection_sort_stable() is False